In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "credit_risk"

split_table = f"{CATALOG}.{SCHEMA}.modeling_credit_default_split"

train_df = (
    spark.table(split_table)
    .filter(F.col("dataset_split") == "train")
)

excluded_columns = {
    "Customer_ID",
    "date_position",
    "flag_default",
    "dataset_split",
}

feature_columns = [
    column
    for column in train_df.columns
    if column not in excluded_columns
]

print(f"Features candidatas: {len(feature_columns)}")
print(feature_columns)

In [0]:
total_rows = train_df.count()
data_types = dict(train_df.dtypes)

profile_expressions = []

for column in feature_columns:
    profile_expressions.extend([
        F.sum(F.col(column).isNull().cast("long")).alias(f"{column}__missing"),
        F.approx_count_distinct(F.col(column)).alias(f"{column}__distinct"),
    ])

profile_result = train_df.agg(*profile_expressions).first().asDict()

profile_rows = [
    (
        column,
        data_types[column],
        profile_result[f"{column}__missing"],
        round(100 * profile_result[f"{column}__missing"] / total_rows, 2),
        profile_result[f"{column}__distinct"],
    )
    for column in feature_columns
]

feature_profile_df = spark.createDataFrame(
    profile_rows,
    [
        "feature_name",
        "data_type",
        "missing_count",
        "missing_pct",
        "approx_distinct_count",
    ],
)

display(
    feature_profile_df
    .orderBy(F.desc("missing_pct"), F.desc("approx_distinct_count"))
)

In [0]:
categorical_columns = [
    column
    for column, data_type in train_df.dtypes
    if data_type == "string" and column in feature_columns
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    
    display(
        train_df
        .groupBy(column)
        .agg(F.count("*").alias("rows"))
        .withColumn(
            "pct",
            F.round(F.col("rows") * 100 / F.lit(total_rows), 2)
        )
        .orderBy(F.desc("rows"))
        .limit(30)
    )

In [0]:
numeric_columns = [
    column
    for column, data_type in train_df.dtypes
    if data_type in {"int", "bigint", "float", "double"}
    and column in feature_columns
]

numeric_summary_expressions = []

for column in numeric_columns:
    numeric_summary_expressions.extend([
        F.min(column).alias(f"{column}__min"),
        F.expr(
            f"percentile_approx(`{column}`, 0.01)"
        ).alias(f"{column}__p01"),
        F.expr(
            f"percentile_approx(`{column}`, 0.50)"
        ).alias(f"{column}__median"),
        F.expr(
            f"percentile_approx(`{column}`, 0.99)"
        ).alias(f"{column}__p99"),
        F.max(column).alias(f"{column}__max"),
    ])

display(train_df.agg(*numeric_summary_expressions))

In [0]:
leakage_risk_columns = {
    "Delay_from_due_date",
    "Payment_Behaviour",
    "Payment_of_Min_Amount",
}

In [0]:
import math
import matplotlib.pyplot as plt

sample_df = (
    train_df
    .select(*numeric_columns)
    .sample(withReplacement=False, fraction=0.15, seed=42)
    .limit(10_000)
    .toPandas()
)

n_columns = 3
n_rows = math.ceil(len(numeric_columns) / n_columns)

fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(18, 4 * n_rows),
)

axes = axes.flatten()

for axis, column in zip(axes, numeric_columns):
    sample_df[column].dropna().plot(
        kind="hist",
        bins=40,
        ax=axis,
        title=column,
    )
    axis.set_xlabel("")
    axis.set_ylabel("Frequência")

for axis in axes[len(numeric_columns):]:
    axis.remove()

plt.tight_layout()
plt.show()

In [0]:
columns_to_check = [
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Num_of_Loan",
    "Num_Credit_Inquiries",
    "Total_EMI_per_month",
]

expressions = []

for column in columns_to_check:
    expressions.extend([
        F.min(column).alias(f"{column}__min"),
        F.max(column).alias(f"{column}__max"),
        F.round(
            100 * F.avg((F.col(column) == 0).cast("int")),
            2,
        ).alias(f"{column}__zero_pct"),
        F.expr(
            f"percentile_approx(`{column}`, array(0.50, 0.90, 0.95, 0.99))"
        ).alias(f"{column}__percentiles"),
    ])

display(train_df.agg(*expressions))

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "credit_risk"

split_table = f"{CATALOG}.{SCHEMA}.modeling_credit_default_split"

valid_ranges = {
    "Num_Bank_Accounts": (0, 50),
    "Num_Credit_Card": (0, 50),
    "Interest_Rate": (0, 100),
    "Num_of_Loan": (0, 20),
}

In [0]:
base_df = spark.table(split_table)

clean_df = base_df

for column, (lower_bound, upper_bound) in valid_ranges.items():
    clean_df = clean_df.withColumn(
        column,
        F.when(
            F.col(column).between(lower_bound, upper_bound),
            F.col(column),
        ).otherwise(F.lit(None))
    )

In [0]:
columns_to_check = [
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Num_of_Loan",
    "Num_Credit_Inquiries",
    "Total_EMI_per_month",
]

expressions = []

for column in columns_to_check:
    expressions.extend([
        F.min(column).alias(f"{column}__min"),
        F.max(column).alias(f"{column}__max"),
        F.round(
            100 * F.avg((F.col(column) == 0).cast("int")),
            2,
        ).alias(f"{column}__zero_pct"),
        F.expr(
            f"percentile_approx(`{column}`, array(0.50, 0.90, 0.95, 0.99))"
        ).alias(f"{column}__percentiles"),
    ])

display(clean_df.agg(*expressions))

In [0]:
safe_categorical_columns = [
    "Credit_Mix",
    "Occupation",
    "Type_of_Loan",
]

for column in safe_categorical_columns:
    clean_df = clean_df.withColumn(
        column,
        F.trim(F.col(column)),
    )

clean_df = clean_df.withColumn(
    "Credit_Mix",
    F.when(F.col("Credit_Mix") == "-", F.lit(None))
     .otherwise(F.col("Credit_Mix")),
)

In [0]:
loan_types_df = (
    clean_df
    .filter(F.col("dataset_split") == "train")
    .select(
        F.explode(
            F.split(
                F.regexp_replace(
                    F.col("Type_of_Loan"),
                    r",\s*and\s+",
                    ",",
                ),
                r"\s*,\s*",
            )
        ).alias("loan_type")
    )
    .withColumn("loan_type", F.trim(F.col("loan_type")))
    .filter(F.col("loan_type").isNotNull() & (F.col("loan_type") != ""))
)

display(
    loan_types_df
    .groupBy("loan_type")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
import re

loan_type_labels = [
    row["loan_type"]
    for row in (
        loan_types_df
        .select("loan_type")
        .distinct()
        .orderBy("loan_type")
        .collect()
    )
]

loan_type_array = F.split(
    F.regexp_replace(
        F.coalesce(F.col("Type_of_Loan"), F.lit("")),
        r",\s*and\s+",
        ",",
    ),
    r"\s*,\s*",
)

prepared_df = clean_df

for loan_type in loan_type_labels:
    safe_name = re.sub(
        r"[^a-z0-9]+",
        "_",
        loan_type.lower(),
    ).strip("_")

    prepared_df = prepared_df.withColumn(
        f"has_{safe_name}",
        F.array_contains(loan_type_array, loan_type).cast("int"),
    )

prepared_df = prepared_df.drop("Type_of_Loan")

In [0]:
loan_flag_columns = [
    column
    for column in prepared_df.columns
    if column.startswith("has_")
]

display(
    prepared_df
    .filter(F.col("dataset_split") == "train")
    .select(
        *[
            F.round(
                100 * F.avg(F.col(column)),
                2,
            ).alias(column)
            for column in loan_flag_columns
        ]
    )
)

In [0]:
categorical_features = [
    "Credit_Mix",
    "Occupation",
]

for column in categorical_features:
    print(f"\n--- {column} ---")

    display(
        prepared_df
        .filter(F.col("dataset_split") == "train")
        .groupBy(column)
        .agg(
            F.count("*").alias("rows"),
            F.round(
                100 * F.avg("flag_default"),
                2,
            ).alias("default_rate_pct"),
        )
        .withColumn(
            "share_of_train_pct",
            F.round(
                100 * F.col("rows") / F.lit(total_rows),
                2,
            ),
        )
        .orderBy(F.desc("rows"))
    )

In [0]:
prepared_df = prepared_df.withColumn(
    "Credit_Mix",
    F.when(
        F.col("Credit_Mix").isin("-", "_"),
        F.lit(None),
    ).otherwise(F.col("Credit_Mix"))
)

In [0]:
excluded_from_baseline = {
    "Customer_ID",
    "date_position",
    "flag_default",
    "dataset_split",
    "Type_of_Loan",
    "Delay_from_due_date",
    "Payment_Behaviour",
    "Payment_of_Min_Amount",
    "Credit_Mix",
}

In [0]:
prepared_table = (
    f"{CATALOG}.{SCHEMA}."
    "modeling_credit_default_feature_base"
)

known_leakage_columns = {
    "Delay_from_due_date",
    "Payment_Behaviour",
    "Payment_of_Min_Amount",
}

columns_to_persist = [
    column
    for column in prepared_df.columns
    if column not in known_leakage_columns
]

feature_base_df = prepared_df.select(*columns_to_persist)

(
    feature_base_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(prepared_table)
)

In [0]:
display(
    feature_base_df
    .groupBy("dataset_split")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("Customer_ID").alias("customers"),
    )
    .orderBy("dataset_split")
)

print(feature_base_df.columns)